### RF Model Training
This noteboook contains the workflow for training the capital and labor intensity random forest models. First, a diagnostic model is trained for model evaluation using spatial CV. Then, a final model is trained using all available data. The output is a set of trained RF models.

In [1]:
import os
from pathlib import Path
import json
from typing import Literal
import dataclasses
from dataclasses import dataclass, field
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from quantile_forest import RandomForestQuantileRegressor


from step_a_config_final import RunConfig
from step_b_spatial_CV_final import make_spatial_folds
from step_c_train_final import train_model

#### Part 1: Train diagnostic models and save results

In [2]:
##### SET-UP DIRECTORIES 

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models_final/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/") 

In [3]:
##### FEATURE COLUMNS
# feature columns were selected using a Recursive Feature Elimination (RFE) approach
# these feature lists were the outcome of that approach 

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_oilcrops_share_base_100_production_USD',
       'rtv_log_pulses_share_base_100_production_USD',
       'rtv_log_roots_tubers_share_base_100_production_USD',
       'rtv_log_sugar_crops_share_base_100_production_USD',
       'rtv_log_ruminants_share_base_100_production_USD']

labor_cols = ['rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_ruminants_share_base_100_production_USD']

In [ ]:
# #### DEFINE RUNS 

# version = 'capital' # capital or labor
# type_of_model = 'rf'  # rf, qrf
# sample ='spatial_CV'

# # auto set-up
# name_of_run =  f'{version}_{type_of_model}_{sample}'
# feature_cols = capital_cols if version == "capital" else labor_cols
# numerator = "USD" if version == "capital" else "jobs"
# target_var = f'rtv_log_{version}_intensity_{numerator}_per_million_tonne' 
# fold = FOLD_DIR / f"{version}_folds.csv"
# model_data = f"{version}_relative_final_thinned.csv"

# RUNS = [
#     RunConfig(
#         run_name         = name_of_run,
#         target           = target_var,   
#         dataset          = model_data,
#         fold_assignments = fold,
#         model_type       = type_of_model,
#         version          = version,
#     ),
# ]

In [4]:
RUNS = [
    RunConfig(
        run_name         = 'capital_rf_spatial_CV',
        target           = 'rtv_log_capital_intensity_USD_per_million_tonne',   
        dataset          = "capital_relative_final_thinned.csv",
        fold_assignments = FOLD_DIR / "capital_folds.csv",
        model_type       = 'rf',
        feature_cols     = capital_cols, 
        version          = 'capital',
    ),
]

In [ ]:
##### RUN   

# function to save results to RESULTS_DIR
def save_results(results, config, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    # config
    config_dict = dataclasses.asdict(config)
    config_dict = {k: str(v) if isinstance(v, Path) else v for k, v in config_dict.items()}
    (out_dir / "config.json").write_text(json.dumps(config_dict, indent=2))

    # combined test + train predictions with fold and split columns
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # best hyperparameters per fold
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

# wrapper function for runs
def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"{'═'*60}")

    df = pd.read_csv(DATA_DIR / config.dataset)

    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, folds, config)

    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)

# run model training
for config in RUNS:
    run(config)


════════════════════════════════════════════════════════════
  run: capital_rf_spatial_CV
════════════════════════════════════════════════════════════

── Spatial folds ────────────────────────────────────────
  fold_1: 18 train countries (1,873 rows) | 1 test countries (1,037 rows)
  fold_2: 18 train countries (1,849 rows) | 1 test countries (1,061 rows)
  fold_3: 18 train countries (2,636 rows) | 1 test countries (274 rows)
  fold_4: 15 train countries (2,640 rows) | 4 test countries (270 rows)
  fold_5: 7 train countries (2,642 rows) | 12 test countries (268 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
